# Annotation Layers

Annotation layers are **reactive visual overlays** rendered on top of existing plots.
A layer is a function that receives a time range and returns a list of annotations
(markers, spans, horizontal lines). When the time range changes, the layer is
automatically re-evaluated and the annotations are updated.

Layers support **knobs** (tunable parameters) just like virtual products.

<div align="center">
<img src="https://github.com/SciQLop/SciQLop/raw/main/SciQLop/resources/icons/SciQLop.png"/>
</div>

## 1. Annotation types

Layers return lists of three annotation types:

| Type | Description | Key fields |
|------|-------------|------------|
| `Marker` | A point on the plot | `time`, `value`, `label`, `color` |
| `Span` | A vertical time interval | `start`, `stop`, `label`, `color` |
| `HLine` | A horizontal reference line | `value`, `label`, `color` |

In [ ]:
from SciQLop.user_api.plot import Marker, Span, HLine

# Examples of annotation objects
m = Marker(time=1e9, value=5.0, label="peak", color="#e74c3c")
s = Span(start=1e9, stop=2e9, label="interval", color="#3498db")
h = HLine(value=0.0, label="baseline", color="#2ecc71")

print(m, s, h)

## 2. The `%%layer` cell magic

The easiest way to create a layer from a notebook. Define a function that takes
`(start, stop)` and returns a list of annotations. The layer appears in the
product tree under **Layers/** — drag it onto any plot.

In [ ]:
%%layer --path "examples/fixed_spans"
import numpy as np

def fixed_spans(start: float, stop: float) -> list[Span]:
    """Highlight fixed 1-hour intervals across the visible range."""
    hour = 3600.0
    t = np.arange(start, stop, hour)
    return [Span(start=t0, stop=t0 + hour / 2, label=f"hour {i}")
            for i, t0 in enumerate(t)]

The `Marker`, `Span`, and `HLine` names are automatically available inside `%%layer` cells.

Re-executing the cell **hot-reloads** the layer — any plot showing it will pick up
the new function on the next time-range change.

## 3. Layers with knobs

Extra keyword arguments with defaults become tunable knobs, exactly like virtual products.
The knobs appear in the inspector panel when the layer is attached to a plot.

In [ ]:
%%layer --path "examples/threshold_crossings"
import numpy as np

def threshold_crossings(
    start: float, stop: float,
    threshold: float = 10.0,
) -> list[Marker | HLine]:
    """Mark points where a sine wave crosses a threshold."""
    t = np.arange(start, stop, 60.0)
    y = 15 * np.sin(2 * np.pi * t / 7200)
    crossings = np.where(np.diff(np.sign(y - threshold)))[0]
    markers = [Marker(time=t[i], value=threshold, label="cross")
               for i in crossings]
    return markers + [HLine(value=threshold, label=f"threshold={threshold}")]

## 4. Programmatic attachment with `PlotPanel.add_layer()`

Instead of drag-and-drop, you can attach a layer directly to a plot from Python.

In [ ]:
import numpy as np
from SciQLop.user_api.plot import create_plot_panel, Span, HLine
from SciQLop.user_api import TimeRange

p = create_plot_panel()
p.time_range = TimeRange("2015-11-18T02:14:30", "2015-11-18T03:34:00")
p.plot("speasy/cda/MMS/MMS1/FGM/MMS1_FGM_SRVY_L2/mms1_fgm_b_gse_srvy_l2")

def b_field_regions(start: float, stop: float,
                    high_threshold: float = 20.0) -> list[HLine]:
    return [
        HLine(value=high_threshold, label="+threshold", color="#e74c3c"),
        HLine(value=-high_threshold, label="-threshold", color="#e74c3c"),
        HLine(value=0.0, label="zero", color="#95a5a6"),
    ]

p.add_layer(b_field_regions, plot_index=0, high_threshold=25.0)

The `plot_index` argument selects which subplot receives the layer (0 = first plot).
Extra keyword arguments set initial knob values.

## 5. The `@register_layer` decorator

For scripts and plugins, use `@register_layer` to register a layer in the product tree
without the cell magic.

In [ ]:
from SciQLop.user_api.layers import register_layer, Span

@register_layer("examples/quiet_intervals")
def quiet_intervals(start: float, stop: float,
                    duration_min: float = 600.0) -> list[Span]:
    """Highlight intervals longer than `duration_min` seconds."""
    import numpy as np
    spans = []
    t = start
    while t + duration_min < stop:
        spans.append(Span(start=t, stop=t + duration_min,
                          color="#27ae60"))
        t += duration_min * 2
    return spans

The layer is now available in the product tree under **Layers/examples/quiet_intervals** —
drag it onto any plot.

## 6. Layers with the fluent API

The fluent builder has a `.layer()` method that attaches a layer to the current subplot.

In [ ]:
import numpy as np
from SciQLop.user_api.plot import fluent, Marker, HLine

def zero_crossings(start: float, stop: float) -> list[Marker | HLine]:
    t = np.arange(start, stop, 60.0)
    y = np.sin(2 * np.pi * t / 3600)
    crossings = np.where(np.diff(np.sign(y)))[0]
    return (
        [Marker(time=t[i], value=0.0, label="zero") for i in crossings]
        + [HLine(value=0.0, color="#7f8c8d")]
    )

panel = (fluent.new_panel()
    .plot("speasy/cda/MMS/MMS1/FGM/MMS1_FGM_SRVY_L2/mms1_fgm_b_gse_srvy_l2")
    .layer(zero_crossings)
    .time_range("2015-11-18T02:14:30", "2015-11-18T03:34:00"))

## 7. Mixing annotation types

A single layer can return a mix of markers, spans, and horizontal lines.

In [ ]:
%%layer --path "examples/mixed_annotations"
import numpy as np

def mixed_annotations(start: float, stop: float) -> list[Marker | Span | HLine]:
    mid = (start + stop) / 2
    quarter = (stop - start) / 4
    return [
        Span(start=mid - quarter, stop=mid + quarter,
             label="center half", color="#f39c12"),
        HLine(value=0.0, label="zero", color="#2ecc71"),
        Marker(time=mid, value=0.0, label="midpoint", color="#e74c3c"),
    ]

## 8. Data-aware layers

Instead of receiving a time range, a layer can receive the **actual data** from a
graph on the same plot. Type-hint the `data` parameter with `Scalar`, `Vector`,
`MultiComponent`, or `Spectrogram` to automatically select the matching graph.

| Type hint | Matches |
|-----------|---------|
| `Scalar` | Line graph with 1 component |
| `Vector` | Line graph with 3+ components |
| `MultiComponent` | Line graph with >1 component |
| `Spectrogram` | Color map |
| *(no hint)* | First available graph |

The callback receives a typed data container with two attributes:
- `data.time` — 1D numpy array of timestamps
- `data.values` — 2D numpy array of shape `(N, ncomponents)`

Data-aware layers are re-evaluated when the **data changes**, not when the time
range changes.

In [ ]:
import numpy as np
from SciQLop.user_api.plot import create_plot_panel, Marker, HLine, Vector
from SciQLop.user_api import TimeRange

p = create_plot_panel()
p.time_range = TimeRange("2015-11-18T02:14:30", "2015-11-18T03:34:00")
p.plot("speasy/cda/MMS/MMS1/FGM/MMS1_FGM_SRVY_L2/mms1_fgm_b_gse_srvy_l2")

def magnitude_peaks(data: Vector, threshold: float = 15.0) -> list[Marker | HLine]:
    """Mark times where the field magnitude exceeds a threshold."""
    mag = data.values[:, -1]  # MMS FGM last column is |B|
    above = np.where(mag > threshold)[0]
    markers = [Marker(time=data.time[i], value=mag[i], label="peak") for i in above[::50]]
    return markers + [HLine(value=threshold, color="#e74c3c")]

p.add_layer(magnitude_peaks, plot_index=0, threshold=20.0)

## API reference

### Annotation types

```python
from SciQLop.user_api.plot import Marker, Span, HLine
# or
from SciQLop.user_api.layers import Marker, Span, HLine
```

| Type | Required fields | Optional fields |
|------|----------------|----------------|
| `Marker(time, value)` | `time: float`, `value: float` | `label`, `color`, `meta` |
| `Span(start, stop)` | `start: float`, `stop: float` | `label`, `color`, `meta` |
| `HLine(value)` | `value: float` | `label`, `color`, `meta` |

### Data type hints

```python
from SciQLop.user_api.plot import Scalar, Vector, MultiComponent, Spectrogram
# or
from SciQLop.user_api.layers import Scalar, Vector, MultiComponent, Spectrogram
```

These serve as both **type hints** and **data containers**. At runtime, the callback
receives an instance with `.time` (1D) and `.values` (2D) attributes.

### Registration

| Method | Use case |
|--------|----------|
| `%%layer` | Notebook cell magic |
| `@register_layer(path)` | Decorator for scripts and plugins |
| `panel.add_layer(func, plot_index=0, **knobs)` | Attach directly to a plot |
| `builder.layer(func, **knobs)` | Fluent API |

### Layer callback signatures

**Range-only** (re-evaluated on time-range change):
```python
def my_layer(start: float, stop: float, **knobs) -> list[Annotation]:
    ...
```

**Data-aware** (re-evaluated on data change):
```python
from SciQLop.user_api.plot import Vector

def my_layer(data: Vector, **knobs) -> list[Annotation]:
    t = data.time          # 1D numpy array of timestamps
    v = data.values        # 2D numpy array, shape (N, ncomponents)
    ...
```

Type-hint `data` with `Scalar`, `Vector`, `MultiComponent`, or `Spectrogram`
to filter which graph provides the data. Omit the hint to use the first available graph.